# Lab 5: Generative Models
*   **Course**: CM52065 Natural Language Processing
*   **Author**: Dr. Andrew Barnes

## Overview
This week we will be exploring three methods of text generation.

The learning objectives for this lab are as follows:

1. Implement an N-Gram generator.
2. Implement a pretrained GPT generator.
3. Compare and contrast temperature settings.

## Preparation

### 1.1 Imports
Let's start by getting all our imports in order.

In [1]:
import nltk
import string
import random
from nltk.corpus import shakespeare
from nltk.util import ngrams
from collections import defaultdict, Counter

nltk.download('gutenberg')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package gutenberg to
[nltk_data]     /Users/kamalrajanisrani/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/kamalrajanisrani/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/kamalrajanisrani/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

### 1.2 Dataset

Now we will load and preprocess a few works of Shakespeare to use in our training regime.

In [2]:
caesar = " ".join(nltk.corpus.gutenberg.words('shakespeare-caesar.txt')).lower()
hamlet = " ".join(nltk.corpus.gutenberg.words('shakespeare-hamlet.txt')).lower()
macbeth = " ".join(nltk.corpus.gutenberg.words('shakespeare-macbeth.txt')).lower()

caesar = caesar.translate(str.maketrans('', '', string.punctuation))
hamlet = hamlet.translate(str.maketrans('', '', string.punctuation))
macbeth = macbeth.translate(str.maketrans('', '', string.punctuation))

text = caesar + hamlet + macbeth

In [3]:
ctok = nltk.word_tokenize(caesar.lower())
htok = nltk.word_tokenize(hamlet.lower())
mtok = nltk.word_tokenize(macbeth.lower())

tokens = ctok + htok + mtok

In [4]:
print(tokens[:100])

['the', 'tragedie', 'of', 'julius', 'caesar', 'by', 'william', 'shakespeare', '1599', 'actus', 'primus', 'scoena', 'prima', 'enter', 'flauius', 'murellus', 'and', 'certaine', 'commoners', 'ouer', 'the', 'stage', 'flauius', 'hence', 'home', 'you', 'idle', 'creatures', 'get', 'you', 'home', 'is', 'this', 'a', 'holiday', 'what', 'know', 'you', 'not', 'being', 'mechanicall', 'you', 'ought', 'not', 'walke', 'vpon', 'a', 'labouring', 'day', 'without', 'the', 'signe', 'of', 'your', 'profession', 'speake', 'what', 'trade', 'art', 'thou', 'car', 'why', 'sir', 'a', 'carpenter', 'mur', 'where', 'is', 'thy', 'leather', 'apron', 'and', 'thy', 'rule', 'what', 'dost', 'thou', 'with', 'thy', 'best', 'apparrell', 'on', 'you', 'sir', 'what', 'trade', 'are', 'you', 'cobl', 'truely', 'sir', 'in', 'respect', 'of', 'a', 'fine', 'workman', 'i', 'am', 'but']


## Part 1: N-Gram Model

The first approach we will take to generating text is through building an N-Gram model using the corpus we loaded above.

To make our lives easier we can use the `ngram` component of the `nltk` library (imported above).

In [5]:
def build_ngram_model(tokens, n):
  model = defaultdict(Counter)
  for gram in ngrams(tokens, n):
    context = gram[:-1]
    next_word = gram[-1]
    model[context][next_word] += 1

  return model

Try this method out below using some trial sets of tokens to get a feel for what it is doing.

In [6]:
model = build_ngram_model(['a', 'b', 'c', 'b', 'a', 'b', 'd', 'a', 'b', 'd', 'c'], 3)

In [7]:
print("Example context:")
example_contexts = list(model.keys())
print(example_contexts)
print("For context bigram: ('a', 'b') we see the following counts " + str(model[example_contexts[0]]))

Example context:
[('a', 'b'), ('b', 'c'), ('c', 'b'), ('b', 'a'), ('b', 'd'), ('d', 'a')]
For context bigram: ('a', 'b') we see the following counts Counter({'d': 2, 'c': 1})


### N-Gram Generator

Now we need to create a generator using the N-Gram model we deveoped above.

See below a sample implementation of a text generator using the model defined above. It continuously samples words from the most recent n-gram until either a unseen sample is reached (OOV) or we reach length.

In [8]:
def generate_text(model, seed, n, length=50):
  current = tuple(seed)
  output = list(seed)
  for _ in range(length):
    if current not in model:
      break
    next_words = model[current]
    words = list(next_words.keys())
    weights = list(next_words.values())
    next_word = random.choices(words, weights=weights)[0]
    output.append(next_word)
    current = tuple(output[-(n-1):])
  return " ".join(output)

See below sample usage of the model defined above to take an input of `a` and `b`. Run this a few times to see how variety is introduced.

### Shakespearian N-Grams

Great, now we have the core functionality it's time to train a Shakespearian N-Gram model to generate text.

In the space below write a script which will do the following tasks:
1. Generate 3 N-Gram models where n = 2, 3 and 4 using the Shakespeare corpus.
2. Generate text using each of these models by passing them the firs `n-1` tokens each.

In [9]:
for n in [2, 3, 4]:
  print(f"== TRAINING {n}-GRAM")
  model = build_ngram_model(tokens, n)
  print(f"== GENERATING USING {n}-GRAM")
  print(generate_text(model, tokens[:n-1], n, length=25))

== TRAINING 2-GRAM
== GENERATING USING 2-GRAM
the images of vs presently ham your selfe there is t the bell blow all the poore and all night i dwell by his stealing out
== TRAINING 3-GRAM
== GENERATING USING 3-GRAM
the tragedie of ivlivs caesar the noble brutus thrusting this report into his sight not pay thee rosse and duncans horses a thing most strange and oftentimes
== TRAINING 4-GRAM
== GENERATING USING 4-GRAM
the tragedie of hamlet prince of denmarke the tragedie of ivlivs caesar the tragedie of macbeth by william shakespeare 1599 actus primus scoena prima thunder and lightning enter


> **Question:** By replaying the generation multiple times, what do you notice about the stability of the outputs?

## Part 2: Transformer Model

In this next section we will introduce ourselves to a new library `transformers` and use this library to load a pre-trained generative model (GPT2) and tokeniser. We will then use this model to generate text.

### Loading the model

To begin, we will load both the model and tokeniser using the `transformers` library.

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

### Generate Text

Now we can use this model to generate text, to do this we use the `model.generate(...)` function which requires us to define a temperature and several other crucial parameters.

Below is an example call to the generate function.

Using the space below modify the prompt, temperature and max_length of the generator and answer the following questions:

> **Question:** What effect does a small temperature have on the output?
> **Question:** What effect does a high temperature have on the output?
> **Question:** Does max_length effect the style of the output?

In [17]:
#prompt = "Why do cows moo?"
prompt = "Explain the sport of F1"

input_ids = tokenizer(prompt, return_tensors="pt").input_ids

gen_tokens = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.5,
    max_length=150,
)
gen_text = tokenizer.batch_decode(gen_tokens)[0]
print(gen_text)

Explain the sport of F1.

What is the sport of F1?

F1 is a sport of motor racing. It is an exciting game, but it is not the only one.

F1 is the most popular sport in the world.

F1 is a sport of sport, and it is a sport of life. It's a sport of life.

F1 is a sport of life.

F1 is a sport of life.

F1 is a sport of life.

F1 is a sport of life.

F1 is a sport of life.

F1 is a sport of life.

F1 is a sport of life


## Optional Exercise (Challenging)

Using the HuggingFace platform and the `transformers` library select and compare three different generative pretrained transformers on the same prompts and identify their strengths and weaknesses of each.

See [HuggingFace](https://huggingface.co/docs/transformers/).

## Wrap-up
Well done on finishing up this week's lab! I hope you were able to practice what we learnt in the lectures and now have a practical understanding of how to use generative models.